# ACA 1 — Aprendizaje Automático
## Sales Forecasting — Retail Superstore

**CUN · Especialización en Inteligencia Artificial · Aprendizaje Automático**

### Objetivo
Construir y evaluar modelos clásicos de Machine Learning para pronosticar ventas
semanales y desplegar el modelo seleccionado mediante Streamlit.

### Dataset
Sales Forecasting - Retail Superstore, Kaggle.

### Ruta
EDA → calidad de datos → transformación temporal → feature engineering →
baseline → modelos clásicos → validación temporal → optimización → pronóstico →
despliegue.


## 1. Carga de librerías y datos

**¿Por qué?** Primero necesitamos herramientas para manipular datos, calcular
métricas, entrenar modelos y visualizar resultados.

In [ ]:
from pathlib import Path
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import TimeSeriesSplit, GridSearchCV

DATA_PATH = Path("../data/train.csv")
df = pd.read_csv(DATA_PATH)

print("Dimensiones:", df.shape)
display(df.head())


## 2. Calidad de datos

Se verifican tipos, nulos y duplicados.

**Resultado:** 9.800 registros, 18 variables, 11 nulos en `Postal Code` y
0 duplicados exactos.

In [ ]:
display(df.dtypes)
print("Duplicados exactos:", df.duplicated().sum())
display(df.isna().sum().sort_values(ascending=False).head(10))


## 3. Fechas y agregación semanal

Las fechas se interpretan como día/mes/año. Para forecasting agregamos las
ventas por semana calendario.

**Resultado:** período 2015-01-03 a 2018-12-30 y 209 semanas.

In [ ]:
df["Order Date"] = pd.to_datetime(df["Order Date"], dayfirst=True, errors="coerce")
df["Sales"] = pd.to_numeric(df["Sales"], errors="coerce")

weekly = (
    df.dropna(subset=["Order Date", "Sales"])
      .set_index("Order Date")["Sales"]
      .resample("W").sum()
      .reset_index()
      .rename(columns={"Order Date": "Date"})
)

print("Fecha inicial:", df["Order Date"].min().date())
print("Fecha final:", df["Order Date"].max().date())
print("Semanas:", len(weekly))
display(weekly.head())


## 4. Análisis exploratorio

La gráfica permite observar el comportamiento de las ventas a través del tiempo.

In [ ]:
plt.figure(figsize=(12, 5))
plt.plot(weekly["Date"], weekly["Sales"])
plt.title("Ventas semanales — Retail Superstore")
plt.xlabel("Fecha")
plt.ylabel("Ventas")
plt.grid(alpha=0.2)
plt.show()


## 5. Feature engineering

Se crean variables de calendario, rezagos y medias móviles.

- Rezagos: 1, 2, 4, 8, 13, 26 y 52 semanas.
- Medias móviles: 4, 8 y 13 semanas.
- Para la Regresión Lineal se agregan variables cíclicas de mes y semana.

Las medias móviles se calculan sobre `Sales.shift(1)` para evitar leakage.

In [ ]:
basic = weekly.copy()
basic["year"] = basic["Date"].dt.year
basic["month"] = basic["Date"].dt.month
basic["quarter"] = basic["Date"].dt.quarter
basic["weekofyear"] = basic["Date"].dt.isocalendar().week.astype(int)

for lag in [1, 2, 4, 8, 13, 26, 52]:
    basic[f"lag_{lag}"] = basic["Sales"].shift(lag)

for window in [4, 8, 13]:
    basic[f"rolling_mean_{window}"] = basic["Sales"].shift(1).rolling(window).mean()

RF_FEATURES = [
    "year","month","quarter","weekofyear",
    "lag_1","lag_2","lag_4","lag_8","lag_13","lag_26","lag_52",
    "rolling_mean_4","rolling_mean_8","rolling_mean_13"
]

cyclic = basic.copy()
cyclic["month_sin"] = np.sin(2*np.pi*cyclic["month"]/12)
cyclic["month_cos"] = np.cos(2*np.pi*cyclic["month"]/12)
cyclic["week_sin"] = np.sin(2*np.pi*cyclic["weekofyear"]/52)
cyclic["week_cos"] = np.cos(2*np.pi*cyclic["weekofyear"]/52)

CYCLIC_FEATURES = [
    "year","month","quarter","weekofyear",
    "month_sin","month_cos","week_sin","week_cos",
    "lag_1","lag_2","lag_4","lag_8","lag_13","lag_26","lag_52",
    "rolling_mean_4","rolling_mean_8","rolling_mean_13"
]

model_data = basic.dropna(subset=RF_FEATURES + ["Sales"]).reset_index(drop=True)
cyclic_data = cyclic.dropna(subset=CYCLIC_FEATURES + ["Sales"]).reset_index(drop=True)

print("Observaciones modelables:", len(model_data))


## 6. Separación cronológica

No se mezclan aleatoriamente las observaciones.

**Resultado:** 125 observaciones de entrenamiento y 32 de prueba.

Esto simula mejor el escenario real: entrenar con el pasado y evaluar sobre un
período posterior.

In [ ]:
train = model_data.iloc[:125]
test = model_data.iloc[125:]

train_c = cyclic_data.iloc[:125]
test_c = cyclic_data.iloc[125:]

print("Train:", len(train))
print("Test:", len(test))


## 7. Baseline

El baseline utiliza la venta de la semana anterior (`lag_1`).

**Resultado:** MAE = $6.916,75; RMSE = $7.914,52; R² = -0,0930.

Este resultado sirve como punto de comparación.

In [ ]:
def metrics(y, p):
    return {
        "MAE": mean_absolute_error(y, p),
        "RMSE": np.sqrt(mean_squared_error(y, p)),
        "R2": r2_score(y, p)
    }

baseline = metrics(test["Sales"], test["lag_1"])
baseline


## 8. Regresión Lineal con variables cíclicas

Las variables seno/coseno representan la naturaleza circular del calendario.

**Resultado obtenido previamente:** MAE = $5.032,24; RMSE = $6.285,68;
R² = 0,3106.

In [ ]:
lr = LinearRegression()
lr.fit(train_c[CYCLIC_FEATURES], train_c["Sales"])
lr_metrics = metrics(test_c["Sales"], lr.predict(test_c[CYCLIC_FEATURES]))
lr_metrics


## 9. Random Forest inicial

Se entrena un Random Forest como modelo no lineal de referencia.

In [ ]:
rf_initial = RandomForestRegressor(
    n_estimators=200, random_state=42, n_jobs=-1
)
rf_initial.fit(train[RF_FEATURES], train["Sales"])
rf_initial_metrics = metrics(
    test["Sales"], rf_initial.predict(test[RF_FEATURES])
)
rf_initial_metrics


## 10. Gradient Boosting

Se incluye un segundo modelo de ensamble para ampliar la comparación de
algoritmos clásicos.

In [ ]:
gb = GradientBoostingRegressor(random_state=42)
gb.fit(train[RF_FEATURES], train["Sales"])
gb_metrics = metrics(test["Sales"], gb.predict(test[RF_FEATURES]))
gb_metrics


## 11. Optimización de Random Forest

Se usa `TimeSeriesSplit(n_splits=5)` en lugar de K-Fold aleatorio.

**¿Por qué?** Porque en una serie temporal el futuro no debe mezclarse con el
pasado durante la validación.

In [ ]:
param_grid = {
    "n_estimators": [200, 500],
    "max_depth": [5, 10, None],
    "max_features": [0.7, 1.0],
    "min_samples_leaf": [1, 2, 4]
}

search = GridSearchCV(
    RandomForestRegressor(random_state=42, n_jobs=-1),
    param_grid,
    cv=TimeSeriesSplit(n_splits=5),
    scoring="neg_mean_absolute_error",
    n_jobs=-1,
    refit=True
)

search.fit(train[RF_FEATURES], train["Sales"])

print("Mejores parámetros:", search.best_params_)
print("MAE CV:", -search.best_score_)


## 12. Evaluación final

El Random Forest optimizado obtuvo:

- **MAE:** $5.349,13
- **RMSE:** $6.675,79
- **R²:** 0,2224
- **MAE CV temporal:** $4.196,67

Hiperparámetros:
`n_estimators=500`, `max_depth=5`, `max_features=0.7`,
`min_samples_leaf=1`.

In [ ]:
optimized = search.best_estimator_
optimized_metrics = metrics(
    test["Sales"], optimized.predict(test[RF_FEATURES])
)
optimized_metrics


## 13. Comparación

| Modelo | MAE | RMSE | R² |
|---|---:|---:|---:|
| Baseline | $6.916,75 | $7.914,52 | -0,0930 |
| Regresión Lineal + cíclicas | $5.032,24 | $6.285,68 | 0,3106 |
| Random Forest optimizado | $5.349,13 | $6.675,79 | 0,2224 |

La Regresión Lineal tiene el mejor desempeño en este bloque de test. Sin embargo,
Random Forest fue seleccionado para despliegue por su mejor desempeño promedio
en validación temporal y por permitir modelar relaciones no lineales.

In [ ]:
comparison = pd.DataFrame([
    ["Baseline", baseline["MAE"], baseline["RMSE"], baseline["R2"]],
    ["Linear Regression + cyclic", lr_metrics["MAE"], lr_metrics["RMSE"], lr_metrics["R2"]],
    ["Random Forest optimized", optimized_metrics["MAE"], optimized_metrics["RMSE"], optimized_metrics["R2"]],
], columns=["Modelo","MAE","RMSE","R2"])
display(comparison)


## 14. Real vs. predicho

In [ ]:
pred = optimized.predict(test[RF_FEATURES])

plt.figure(figsize=(12,5))
plt.plot(test["Date"], test["Sales"], label="Real")
plt.plot(test["Date"], pred, label="Predicción")
plt.title("Random Forest — Real vs. Predicho")
plt.xlabel("Fecha")
plt.ylabel("Ventas")
plt.legend()
plt.grid(alpha=0.2)
plt.show()


## 15. Importancia de variables

In [ ]:
importance = pd.DataFrame({
    "Variable": RF_FEATURES,
    "Importancia": optimized.feature_importances_
}).sort_values("Importancia", ascending=False)

display(importance)

plt.figure(figsize=(9,5))
top = importance.head(10).sort_values("Importancia")
plt.barh(top["Variable"], top["Importancia"])
plt.title("Top 10 variables")
plt.xlabel("Importancia")
plt.show()


## 16. Reentrenamiento y guardado

Una vez seleccionados los hiperparámetros, el modelo final se reentrena con las
157 observaciones disponibles para modelado. Luego se guarda para Streamlit.

In [ ]:
from pathlib import Path
import joblib

MODELS = Path("../models")
MODELS.mkdir(exist_ok=True)

final_model = RandomForestRegressor(
    **search.best_params_, random_state=42, n_jobs=-1
)
final_model.fit(model_data[RF_FEATURES], model_data["Sales"])

joblib.dump(final_model, MODELS / "sales_forecasting_rf.pkl")
print("Guardado:", MODELS / "sales_forecasting_rf.pkl")


## 17. Conclusiones

1. El dataset contiene 9.800 registros y 18 variables.
2. La transformación semanal produjo 209 semanas y 157 observaciones utilizables
   después de generar rezagos y medias móviles.
3. La validación debe respetar el orden temporal.
4. La Regresión Lineal con variables cíclicas obtuvo el mejor resultado en el
   bloque test, mientras Random Forest mostró el mejor MAE promedio durante la
   validación temporal utilizada para seleccionar hiperparámetros.
5. El Random Forest optimizado se utiliza como modelo de despliegue en Streamlit.
6. El pronóstico debe interpretarse como una estimación estadística y no como
   garantía de ventas futuras.
